# Environment Smoke Test
Tests SpatialMETA, Scanpy, and Squidpy installation. All plots are saved to `./smoke_test_plots/` instead of displayed, for WSL compatibility.

In [1]:
# Cell 0 - WSL matplotlib setup (local only, not needed on Linux server)
import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import matplotlib.pyplot as plt
from IPython.display import display
import os

# All plots will be saved here
PLOT_DIR = './smoke_test_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(filename):
    """Save current figure and close it."""
    path = os.path.join(PLOT_DIR, filename)
    plt.savefig(path, bbox_inches='tight', dpi=100)
    plt.close()
    print(f'  Plot saved: {path}')

print('✓ Matplotlib configured (Agg backend, plots saving to', PLOT_DIR, ')')

✓ Matplotlib configured (Agg backend, plots saving to ./smoke_test_plots )


In [2]:
# Cell 1 - Version check
import scanpy as sc
import squidpy as sq
import spatialmeta as smt

sc.logging.print_header()
print(f'squidpy=={sq.__version__}')
print(f'spatialmeta=={smt.__version__}')
print('\n✓ All three packages imported successfully')

scanpy==1.9.1 anndata==0.8.0 umap==0.5.1 numpy==1.24.4 scipy==1.9.1 pandas==1.5.3 scikit-learn==1.5.0 statsmodels==0.14.1 python-igraph==1.0.0 leidenalg==0.11.0 pynndescent==0.6.0
squidpy==1.2.2
spatialmeta==0.0.3.0

✓ All three packages imported successfully


In [3]:
# Cell 2 - Scanpy smoke test
print('--- Scanpy smoke test ---')

# Downloads ~50MB on first run, cached after that
adata = sc.datasets.visium_sge(sample_id='V1_Human_Lymph_Node')
adata.var_names_make_unique()
print(f'  Loaded: {adata.shape[0]} spots x {adata.shape[1]} genes')

# Preprocessing pipeline
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)
print('  Preprocessing pipeline complete')

# Plot 1: PCA variance ratio
sc.pl.pca_variance_ratio(adata, n_pcs=15, show=False)
save_plot('scanpy_pca_variance_ratio.png')

# Plot 2: UMAP with leiden clusters
sc.pl.umap(adata, color='leiden', show=False)
save_plot('scanpy_umap_leiden.png')

# Plot 3: Spatial plot
sc.pl.spatial(adata, color='leiden', show=False)
save_plot('scanpy_spatial_leiden.png')

print(f'\n✓ Scanpy OK — {adata.shape[0]} spots processed, 3 plots saved')

--- Scanpy smoke test ---
  Loaded: 4035 spots x 36601 genes


findfont: Font family ['arial'] not found. Falling back to DejaVu Sans.
findfont: Font family ['arial'] not found. Falling back to DejaVu Sans.
findfont: Font family ['arial'] not found. Falling back to DejaVu Sans.
findfont: Font family ['arial'] not found. Falling back to DejaVu Sans.


  Preprocessing pipeline complete
  Plot saved: ./smoke_test_plots/scanpy_pca_variance_ratio.png
  Plot saved: ./smoke_test_plots/scanpy_umap_leiden.png
  Plot saved: ./smoke_test_plots/scanpy_spatial_leiden.png

✓ Scanpy OK — 4035 spots processed, 3 plots saved


In [5]:
# Cell 3 - Squidpy smoke test
print('--- Squidpy smoke test ---')

adata_sq = sq.datasets.visium_hne_adata()
print(f'  Loaded: {adata_sq.shape[0]} spots x {adata_sq.shape[1]} genes')

# Spatial graph and neighbourhood enrichment
sq.gr.spatial_neighbors(adata_sq, coord_type='grid')
sq.gr.nhood_enrichment(adata_sq, cluster_key='cluster')
print('  Spatial graph and neighbourhood enrichment complete')

# Plot 1: Neighbourhood enrichment
sq.pl.nhood_enrichment(adata_sq, cluster_key='cluster', figsize=(5, 4), show=False)
save_plot('squidpy_nhood_enrichment.png')

# Plot 2: Spatial scatter
sq.pl.spatial_scatter(adata_sq, color='cluster')
save_plot('squidpy_spatial_scatter.png')

print(f'\n✓ Squidpy OK — spatial graph built, 2 plots saved')

--- Squidpy smoke test ---
  Loaded: 2688 spots x 18078 genes


  0%|          | 0/1000 [00:00<?, ?/s]

  Spatial graph and neighbourhood enrichment complete
  Plot saved: ./smoke_test_plots/squidpy_nhood_enrichment.png
  Plot saved: ./smoke_test_plots/squidpy_spatial_scatter.png

✓ Squidpy OK — spatial graph built, 2 plots saved


In [12]:
help(smt.data.load_adata)


Help on function load_adata in module spatialmeta.data._dataloader:

load_adata(sample_name: str, modality: Literal['ST', 'SM', 'joint']) -> Union[spatialmeta.util._classes.AnnDataST, spatialmeta.util._classes.AnnDataSM, spatialmeta.util._classes.AnnDataJointSMST]
    Load the AnnData object for the given sample name and modality.
    
    :param sample_name: str
        The name of the sample. Use `list_datasets` to get the list of all available datasets.
    :param modality: Literal["ST", "SM", "joint"]
        The modality of the dataset. Choose from "ST", "SM", or "joint".



In [25]:
# Cell 4 - SpatialMETA smoke test (paper dataset)
print('--- SpatialMETA smoke test ---')

# List available datasets from the paper
print('  Available datasets:')
print(smt.data.list_datasets())

# Load the ST (spatial transcriptomics) AnnData
# Adjust dataset key to match list_datasets() output above
adata_joint = smt.data.load_adata(sample_name='Y7_T_raw', modality='joint')
print(f'  Loaded ST+SM data: {adata_st.shape}')

# Plot: first gene in spatial context
sc.pl.spatial(adata_st, color=adata_st.var_names[0], show=False)
save_plot('spatialmeta_ST_first_gene.png')

print(f'\n✓ SpatialMETA OK — paper dataset loaded, 1 plot saved')

--- SpatialMETA smoke test ---
  Available datasets:
                                                  url  \
0   /records/14986870/files/adata_joint_R114_T_raw...   
1   /records/14986870/files/adata_joint_S15_T_raw....   
2   /records/14986870/files/adata_joint_X49_T_raw....   
3   /records/14986870/files/adata_joint_Y27_T_raw....   
4   /records/14986870/files/adata_joint_Y7_T_raw.h...   
5   /records/14986870/files/adata_SM_R114_T_raw.h5...   
6   /records/14986870/files/adata_SM_S15_T_raw.h5a...   
7   /records/14986870/files/adata_SM_X49_T_raw.h5a...   
8   /records/14986870/files/adata_SM_Y27_T_raw.h5a...   
9   /records/14986870/files/adata_SM_Y7_T_raw.h5ad...   
10  /records/14986870/files/adata_SM_Y7_T_raw.alig...   
11  /records/14986870/files/adata_ST_Y7_T_raw.h5ad...   
12  /records/14986870/files/mouse3_brain_msi.csv?d...   
13  /records/14986870/files/results_data.zip?downl...   
14    /records/14986870/files/Y7_T_neg.ibd?download=1   
15  /records/14986870/files/Y7_T_ne

In [29]:
# Cell 4b - SpatialMETA minimal pipeline test
print('--- SpatialMETA pipeline test ---')

import spatialmeta as smt
import scanpy as sc

# Use already loaded joint object
adata = adata_joint.copy()  # work on a copy to preserve the original
print(f'  Input: {adata.shape}')
print(f'  Obs columns: {list(adata.obs.columns[:5])}')  # peek at metadata
print(f'  Obsm keys: {list(adata.obsm.keys())}')        # check spatial coords exist

# Step 1: Normalise
smt.pp.normalize_total_joint_adata_sm_st(adata)
print('  ✓ Normalisation complete')

# Step 2: Find spatially variable features
smt.pp.spatial_variable_joint_adata_sm_st(adata)
print('  ✓ Spatially variable features identified')

# Step 3: Integration model
model = smt.model.ConditionalVAESTSM(adata)
model.fit(max_epoch=10)  # 10 epochs just to confirm it runs, not for real results
print('  ✓ Model trained')

# Step 4: Get latent embedding
adata.obsm['X_spatialmeta'] = model.get_latent_embedding()
print(f'  ✓ Latent representation shape: {adata.obsm["X_spatialmeta"].shape}')

# Step 5: Cluster on the latent space
sc.pp.neighbors(adata, use_rep='X_spatialmeta')
sc.tl.leiden(adata, resolution=0.5)
print(f'  ✓ Clustering complete — {adata.obs["leiden"].nunique()} clusters found')

# Plot: spatial map of clusters
sc.pl.spatial(adata, color='leiden', show=False)
save_plot('spatialmeta_pipeline_clusters.png')

print('\n✓ SpatialMETA pipeline OK')

--- SpatialMETA pipeline test ---
  Input: (1914, 37703)
  Obs columns: ['x_coord', 'y_coord', 'spot_name', 'total_intensity', 'mean_intensity']
  Obsm keys: ['spatial']
  ✓ Normalisation complete
  ✓ Spatially variable features identified


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

  ✓ Model trained


Latent Embedding:   0%|          | 0/15 [00:00<?, ?it/s]

  ✓ Latent representation shape: (1914, 10)
  ✓ Clustering complete — 7 clusters found
  Plot saved: ./smoke_test_plots/spatialmeta_pipeline_clusters.png

✓ SpatialMETA pipeline OK


In [22]:
# Cell 5 - Cross-package integration check
print('--- Cross-package integration check ---')

adata_combined = sq.datasets.visium_hne_adata_crop()

# Pass a Squidpy-loaded object through a full Scanpy pipeline
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)
sc.pp.pca(adata_combined, n_comps=10)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)
print('  Scanpy pipeline on Squidpy data complete')

# Plot: UMAP coloured by cluster
sc.pl.umap(adata_combined, color='cluster', show=False)
save_plot('integration_umap.png')

print(f'\n✓ Scanpy <-> Squidpy interop OK, 1 plot saved')

--- Cross-package integration check ---


  0%|          | 0.00/89.9M [00:00<?, ?B/s]

  Scanpy pipeline on Squidpy data complete
  Plot saved: ./smoke_test_plots/integration_umap.png

✓ Scanpy <-> Squidpy interop OK, 1 plot saved


In [23]:
# Cell 6 - Summary
import glob

plots = glob.glob(os.path.join(PLOT_DIR, '*.png'))
print('=== Smoke Test Complete ===')
print(f'All plots saved to: {PLOT_DIR}/')
print(f'Total plots generated: {len(plots)}')
for p in sorted(plots):
    size_kb = os.path.getsize(p) / 1024
    print(f'  {os.path.basename(p)} ({size_kb:.1f} KB)')

=== Smoke Test Complete ===
All plots saved to: ./smoke_test_plots/
Total plots generated: 7
  integration_umap.png (78.0 KB)
  scanpy_pca_variance_ratio.png (18.7 KB)
  scanpy_spatial_leiden.png (297.0 KB)
  scanpy_umap_leiden.png (150.7 KB)
  spatialmeta_ST_first_gene.png (300.8 KB)
  squidpy_nhood_enrichment.png (54.2 KB)
  squidpy_spatial_scatter.png (42.2 KB)
